In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os

csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


# data=pd.read_csv(path)
# data.head()

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:

import matplotlib.pyplot as plt

# target_column= df['Delivery_Time']
# df[target_column].hist(bins=30, edgecolor='black')
# #df[target_column].hist()

# plt.title(f"Target Distribution ({target_column})")
# plt.xlabel(target_column)
# plt.ylabel("Frequency")
# plt.grid(False)
# plt.show()

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df

In [ ]:
# Task 1: Write your code here:
df=df.drop('Order_ID',axis=1)
df

In [ ]:
# Task 2: Write your code here:
df.isna().sum()


In [ ]:
#this four columns have missing but not that much, so better solution to remove it
cols=['Weather','Traffic_Level','Time_of_Day','Courier_Experience_yrs']
df=df.dropna(subset=cols)
df

In [ ]:
#imprtant column and have large missing values, so fill nul be median to prevent outlire
df['Delivery_Time']=df['Delivery_Time'].fillna(df['Delivery_Time'].median())

In [ ]:
df.isna().sum()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df

In [ ]:
# Task 4 Encoding : Write your code here:
from sklearn.preprocessing import OneHotEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns

encoder=OneHotEncoder(sparse_output=False)
onehot=encoder.fit_transform(df[categorical_cols])

new_cols=encoder.get_feature_names_out(categorical_cols)
one_hot_df=pd.DataFrame(onehot,columns=new_cols)

df_en = pd.concat([df.drop(columns=categorical_cols), one_hot_df], axis=1)
df_en.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 5 StandardScaler: Write your code here:
from sklearn.preprocessing import StandardScaler


features = df_en.columns.drop("Delivery_Time")

scaler=StandardScaler()

df_en[features]= scaler.fit_transform(df_en.drop('Delivery_Time',axis=1))

df_en.head()

In [ ]:
# Task 6: Write your code here:
df_en['Delivery_Time'].value_counts(normalize=True)

In [ ]:
# Task 1: Write your code here:

X = df_en.drop("Delivery_Time", axis=1)
y = df_en['Delivery_Time']


In [ ]:
df_en.isna().sum()

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold,cross_val_score
from sklearn.metrics import mean_absolute_error
import numpy as np

results = []
n_splits=5
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models={"RandomForestRegressor": RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)}


for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    results.append(r2)


print(f"  R2:    {np.mean(results):.4f}")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'][:15], feature_importance['importance'][:15])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: